In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 4.2 MB/s eta 0:00:00


In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [3]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("Open_AI")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://bolob435.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [4]:
# 1. Define the input data
medical_report = """
Patient Name: John Doe

Hemoglobin: 10 g/dL
WBC Count: 12000 /µL
Platelets: 150000 /µL

Observation:
Patient shows signs of mild anemia and possible infection.
"""

# 2. Use the Responses API for structured extraction
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """Analyze the following medical report and return ONLY valid JSON with:

                        - Report Type
                        - Key Findings
                        - Explanation
                        - Abnormal Indicators
                        - Possible Interpretation
                        - Summary
                        """
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_report}"
                    }
                ]
            }
        ]
    )

    print("--- INTERPRETED REPORT ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Error: {e}")

--- INTERPRETED REPORT ---
[ResponseOutputText(annotations=[], text='{\n  "Report Type": "Complete Blood Count (CBC)",\n  "Key Findings": {\n    "Hemoglobin": "10 g/dL",\n    "WBC Count": "12000 /µL",\n    "Platelets": "150000 /µL"\n  },\n  "Explanation": {\n    "Hemoglobin": "Normal adult ranges are typically 13.5-17.5 g/dL for men; this value is low, indicating anemia.",\n    "WBC Count": "Normal range is 4000-11000 /µL; this value is slightly elevated, suggesting possible infection or inflammation.",\n    "Platelets": "Normal range is 150000-450000 /µL; this value is within normal limits."\n  },\n  "Abnormal Indicators": [\n    "Low hemoglobin (anemia)",\n    "Elevated WBC count (possible infection)"\n  ],\n  "Possible Interpretation": [\n    "Mild anemia",\n    "Possible underlying infection"\n  ],\n  "Summary": "The patient demonstrates mild anemia and a mildly elevated white blood cell count, indicating a potential infection. Platelet count is normal."\n}', type='output_text', lo

In [5]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Report Type": "Complete Blood Count (CBC)",
  "Key Findings": {
    "Hemoglobin": "10 g/dL",
    "WBC Count": "12000 /\u00b5L",
    "Platelets": "150000 /\u00b5L"
  },
  "Explanation": {
    "Hemoglobin": "Normal adult ranges are typically 13.5-17.5 g/dL for men; this value is low, indicating anemia.",
    "WBC Count": "Normal range is 4000-11000 /\u00b5L; this value is slightly elevated, suggesting possible infection or inflammation.",
    "Platelets": "Normal range is 150000-450000 /\u00b5L; this value is within normal limits."
  },
  "Abnormal Indicators": [
    "Low hemoglobin (anemia)",
    "Elevated WBC count (possible infection)"
  ],
  "Possible Interpretation": [
    "Mild anemia",
    "Possible underlying infection"
  ],
  "Summary": "The patient demonstrates mild anemia and a mildly elevated white blood cell count, indicating a potential infection. Platelet count is normal."
}


In [7]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=mrinalstorage;AccountKey=PHtgo0Sd+nYgLm4WXvHBp+rv7laJJsvg6Y3EXfO4ZEYuNDlffd2RdauObtclMrVVbxHpF6o8QmKN+ASt0qj8PA==;EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "medical"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [8]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [9]:
blob_client = blob_service_client.get_blob_client(
    container="medical",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F78D8DFD427"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 37, 31, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xd1\xc073\xe4\xd8\x1d\xc1]I\x99\xb1f\xbaY\xba'),
 'client_request_id': 'f49c7f56-3d54-11f1-b91d-0242ac1c000c',
 'request_id': '0b51d9fa-701e-00a3-0261-d1eea2000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 37, 31, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [10]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_073723.txt
